# 03 Aria MPS Visualization

This notebook visualizes the 3D trajectory and semidense point cloud from a Project Aria Gen 1
recording using `rerun-sdk` and `projectaria_tools`.


The goal is to compare the quality of the point cloud generated by:
1. A recording with only ambient light vs a recording with additional light sources
scrivi che il secondo confrontro è per vedere se si geenra meglio la point cloud se tengo la telecamera fissa su uno stesso punto o se la muovo per catturare più dettagli.
2. A recording with a fixed camera position vs a recording where the camera is moved to capture more details

#### Premises:
The recordings being compared were made in similar situations, trying to recreate the same path and of the same length to try to have a more fair comparison.

#### Note:
This build exposes only minimal symbols in `rerun_helpers`, so we log trajectory and semidense points directly from MPS CSV files using `projectaria_tools.core.mps` readers and `rerun` primitives.

## 3.1 Recording Configuration

Define input paths for the two recordings:
- `gnome_bright`
- `gnome_dark`

Then validate that the expected folders exist.

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import rerun as rr

from projectaria_tools.core import mps

In [2]:
BRIGHT_VRS = os.path.join("..", "data", "raw", "gnome_bright", "gnome_bright.vrs")
BRIGHT_MPS = os.path.join("..", "data", "raw", "gnome_bright", "mps_gnome_bright_vrs")
BRIGHT_SLAM = os.path.join(BRIGHT_MPS, "slam")

DARK_VRS = os.path.join("..", "data", "raw", "gnome_dark", "gnome_dark.vrs")
DARK_MPS = os.path.join("..", "data", "raw", "gnome_dark", "mps_gnome_dark_vrs")
DARK_SLAM = os.path.join(DARK_MPS, "slam")

def assert_exists(path: str, label: str):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{label} not found: {os.path.abspath(path)}")

assert_exists(BRIGHT_VRS, "BRIGHT VRS")
assert_exists(BRIGHT_SLAM, "BRIGHT SLAM folder")
assert_exists(DARK_VRS, "DARK VRS")
assert_exists(DARK_SLAM, "DARK SLAM folder")

print("All input paths are valid.")

All input paths are valid.


## 3.3 Initialize Rerun

Start a Rerun session for 3D visualization.

In [3]:
rr.init("Aria_Gnome_Comparison", spawn=True)
print("Rerun initialized.")

Rerun initialized.


## 3.4 Data Loading Helpers

These helpers:
- locate trajectory/point-cloud CSV files,
- load trajectory using MPS APIs,
- load point cloud using multiple reader variants across library builds,
- fallback to CSV parsing if needed.

In [4]:
def find_trajectory_file(slam_dir: str) -> str:
    candidates = sorted(
        glob.glob(os.path.join(slam_dir, "closed_loop_trajectory.csv")) +
        glob.glob(os.path.join(slam_dir, "closed_loop_trajectory.csv.gz"))
    )
    if not candidates:
        raise FileNotFoundError(f"No closed_loop_trajectory file found in: {slam_dir}")
    return candidates[0]

In [5]:
def find_points_file(slam_dir: str) -> str:
    candidates = sorted(
        glob.glob(os.path.join(slam_dir, "semidense_points.csv")) +
        glob.glob(os.path.join(slam_dir, "semidense_points.csv.gz"))
    )
    if not candidates:
        raise FileNotFoundError(f"No semidense_points file found in: {slam_dir}")
    return candidates[0]

In [6]:
def load_points_any(pts_path: str):
    reader_candidates = [
        "read_semidense_points",
        "read_global_point_cloud",
        "read_point_cloud",
        "read_global_points",
        "read_semidense_point_cloud",
    ]

    for reader_name in reader_candidates:
        if hasattr(mps, reader_name):
            reader = getattr(mps, reader_name)
            if callable(reader):
                try:
                    data = reader(pts_path)
                    return {"mode": "mps_reader", "reader": reader_name, "data": data}
                except Exception:
                    pass

    # CSV fallback
    df = pd.read_csv(pts_path)
    xyz_candidates = [
        ("x", "y", "z"),
        ("X", "Y", "Z"),
        ("px_world", "py_world", "pz_world"),
        ("p_x", "p_y", "p_z"),
    ]
    for cx, cy, cz in xyz_candidates:
        if all(c in df.columns for c in (cx, cy, cz)):
            xyz = df[[cx, cy, cz]].to_numpy(dtype=np.float32)
            return {"mode": "csv_xyz", "reader": "pandas_fallback", "data": xyz}

    raise RuntimeError(f"Could not extract XYZ columns from: {pts_path}")

## 3.5 Geometry Extraction Helpers

Convert trajectory poses and point-cloud records to clean `Nx3` NumPy arrays.

In [7]:
def pose_to_xyz(pose_obj):
    for name in ["transform_world_device", "T_world_device", "world_T_device"]:
        if hasattr(pose_obj, name):
            t = getattr(pose_obj, name)
            t = t() if callable(t) else t
            if hasattr(t, "translation"):
                v = t.translation()
                v = v() if callable(v) else v
                arr = np.asarray(v, dtype=np.float64).reshape(-1)
                if arr.size >= 3:
                    return arr[:3]
    return None

def point_to_xyz(point_obj):
    for name in ["position_world", "position", "p_w", "xyz"]:
        if hasattr(point_obj, name):
            v = getattr(point_obj, name)
            v = v() if callable(v) else v
            arr = np.asarray(v, dtype=np.float64).reshape(-1)
            if arr.size >= 3:
                return arr[:3]
    return None

def to_xyz_arrays(traj_raw, pts_result):
    traj_xyz = np.array([x for x in (pose_to_xyz(p) for p in traj_raw) if x is not None], dtype=np.float32)

    if pts_result["mode"] == "mps_reader":
        pts_raw = pts_result["data"]
        pts_xyz = np.array([x for x in (point_to_xyz(p) for p in pts_raw) if x is not None], dtype=np.float32)
    else:
        pts_xyz = pts_result["data"].astype(np.float32)

    traj_xyz = traj_xyz[np.isfinite(traj_xyz).all(axis=1)]
    pts_xyz = pts_xyz[np.isfinite(pts_xyz).all(axis=1)]

    return traj_xyz, pts_xyz

## 3.6 Visualization Helper

Log a centered scene to Rerun for easier camera navigation.

In [8]:
def log_scene(scene_name: str, traj_xyz: np.ndarray, pts_xyz: np.ndarray, max_points: int = 500000):
    center = np.mean(pts_xyz, axis=0, keepdims=True)
    traj_local = traj_xyz - center
    pts_local = pts_xyz - center

    if pts_local.shape[0] > max_points:
        idx = np.random.choice(pts_local.shape[0], size=max_points, replace=False)
        pts_local = pts_local[idx]

    rr.log(f"{scene_name}/trajectory", rr.LineStrips3D([traj_local]))
    rr.log(f"{scene_name}/points", rr.Points3D(pts_local, radii=0.03))
    rr.log(
        f"{scene_name}/origin",
        rr.Points3D(
            np.array([[0, 0, 0]], dtype=np.float32),
            radii=0.0005,
            colors=np.array([[255, 0, 0]], dtype=np.uint8),
        ),
    )

## 3.7 Load and Visualize Bright and Dark Recording and Compare the Results

In [9]:
bright_traj_path = find_trajectory_file(BRIGHT_SLAM)
bright_pts_path = find_points_file(BRIGHT_SLAM)

bright_traj_raw = mps.read_closed_loop_trajectory(bright_traj_path)
bright_pts_result = load_points_any(bright_pts_path)

bright_traj_xyz, bright_pts_xyz = to_xyz_arrays(bright_traj_raw, bright_pts_result)
log_scene("scene/bright", bright_traj_xyz, bright_pts_xyz)

print("BRIGHT trajectory poses:", len(bright_traj_raw))
print("BRIGHT points reader:", bright_pts_result["reader"])
print("BRIGHT points count:", bright_pts_xyz.shape[0])

BRIGHT trajectory poses: 133179
BRIGHT points reader: read_global_point_cloud
BRIGHT points count: 319326


In [10]:
dark_traj_path = find_trajectory_file(DARK_SLAM)
dark_pts_path = find_points_file(DARK_SLAM)

dark_traj_raw = mps.read_closed_loop_trajectory(dark_traj_path)
dark_pts_result = load_points_any(dark_pts_path)

dark_traj_xyz, dark_pts_xyz = to_xyz_arrays(dark_traj_raw, dark_pts_result)
log_scene("scene/dark", dark_traj_xyz, dark_pts_xyz)

print("DARK trajectory poses:", len(dark_traj_raw))
print("DARK points reader:", dark_pts_result["reader"])
print("DARK points count:", dark_pts_xyz.shape[0])

DARK trajectory poses: 137745
DARK points reader: read_global_point_cloud
DARK points count: 448627


In [11]:
comparison_df = pd.DataFrame([
    {"recording": "bright", "points_count": int(bright_pts_xyz.shape[0]), "trajectory_poses": int(bright_traj_xyz.shape[0])},
    {"recording": "dark",   "points_count": int(dark_pts_xyz.shape[0]),   "trajectory_poses": int(dark_traj_xyz.shape[0])},
])

display(comparison_df)

,recording,points_count,trajectory_poses
0,bright,319326,133179
1,dark,448627,137745


## 3.8 Second Comparison: Static vs Dynamic Camera

This section compares point-cloud quality between:
1. **Static camera** : camera fixed at one position
2. **Dynamic camera** : camera moves to capture more details

The goal is to evaluate whether camera motion improves reconstruction quality and point density.

In [12]:
# Static camera recording paths
STATIC_VRS = os.path.join("..", "data", "raw", "gnome_face_static", "gnome_face_static.vrs")
STATIC_MPS = os.path.join("..", "data", "raw", "gnome_face_static", "mps_gnome_face_static_vrs")
STATIC_SLAM = os.path.join(STATIC_MPS, "slam")

# Dynamic camera recording paths
DYNAMIC_VRS = os.path.join("..", "data", "raw", "gnome_face_dynamic", "gnome_face_dynamic.vrs")
DYNAMIC_MPS = os.path.join("..", "data", "raw", "gnome_face_dynamic", "mps_gnome_face_dynamic_vrs")
DYNAMIC_SLAM = os.path.join(DYNAMIC_MPS, "slam")

# Validate paths
assert_exists(STATIC_VRS, "STATIC VRS")
assert_exists(STATIC_SLAM, "STATIC SLAM folder")
assert_exists(DYNAMIC_VRS, "DYNAMIC VRS")
assert_exists(DYNAMIC_SLAM, "DYNAMIC SLAM folder")

print("All static/dynamic paths are valid.")

All static/dynamic paths are valid.


## 3.9 Load and Visualize Static and Dynamic Recordings and Compare the Results

In [13]:
# Load static camera recording
static_traj_path = find_trajectory_file(STATIC_SLAM)
static_pts_path = find_points_file(STATIC_SLAM)

static_traj_raw = mps.read_closed_loop_trajectory(static_traj_path)
static_pts_result = load_points_any(static_pts_path)

static_traj_xyz, static_pts_xyz = to_xyz_arrays(static_traj_raw, static_pts_result)
log_scene("scene/static", static_traj_xyz, static_pts_xyz)

print("STATIC trajectory poses:", len(static_traj_raw))
print("STATIC points reader:", static_pts_result["reader"])
print("STATIC points count:", static_pts_xyz.shape[0])

STATIC trajectory poses: 29296
STATIC points reader: read_global_point_cloud
STATIC points count: 31459


In [14]:
# Load dynamic camera recording
dynamic_traj_path = find_trajectory_file(DYNAMIC_SLAM)
dynamic_pts_path = find_points_file(DYNAMIC_SLAM)

dynamic_traj_raw = mps.read_closed_loop_trajectory(dynamic_traj_path)
dynamic_pts_result = load_points_any(dynamic_pts_path)

dynamic_traj_xyz, dynamic_pts_xyz = to_xyz_arrays(dynamic_traj_raw, dynamic_pts_result)
log_scene("scene/dynamic", dynamic_traj_xyz, dynamic_pts_xyz)

print("DYNAMIC trajectory poses:", len(dynamic_traj_raw))
print("DYNAMIC points reader:", dynamic_pts_result["reader"])
print("DYNAMIC points count:", dynamic_pts_xyz.shape[0])

DYNAMIC trajectory poses: 37361
DYNAMIC points reader: read_global_point_cloud
DYNAMIC points count: 159204


In [15]:
# Build comparison table: static vs dynamic
static_dynamic_df = pd.DataFrame([
    {
        "recording": "static",
        "points_count": int(static_pts_xyz.shape[0]),
        "trajectory_poses": int(static_traj_xyz.shape[0]),
    },
    {
        "recording": "dynamic",
        "points_count": int(dynamic_pts_xyz.shape[0]),
        "trajectory_poses": int(dynamic_traj_xyz.shape[0]),
    },
])

display(static_dynamic_df)

,recording,points_count,trajectory_poses
0,static,31459,29296
1,dynamic,159204,37361
